### Urlaubsfahrt – Lösung Schritt für Schritt

*38. Bundeswettbewerb Informatik (2019), 1. Runde, Aufgabe 4*

Die Eltern von Justin, Esra, Julia und Vincent machen mit ihnen eine lange Urlaubsreise quer durch Europa mit einem Wohnmobil. Während der langen Fahrt müssen sie immer wieder halten, um zu tanken.

Justin sagt: „Lasst uns darauf achten, nicht so teuer zu tanken.“ Esra dagegen möchte möglichst schnell ankommen und meint: „Wir sind lange unterwegs. Ich möchte daher nicht oft anhalten; so groß sind die Preisunterschiede beim Diesel nun auch wieder nicht. Lass uns jedes Mal volltanken, dann kommen wir schneller an.“

Nach einiger Diskussion macht Julia den Vorschlag: „Wir machen es so: Wir tanken nur so oft, wie es unbedingt nötig ist; dafür suchen wir uns aber die günstigsten Tankstellen raus.“

Vincent hat in der Zwischenzeit mittels einer App auf seinem Smartphone alle auf der Reiseroute liegenden Tankstellen mit ihren Entfernungen und Preisen für Diesel herausgefunden.

#### Aufgabe 4

Schreibe ein Programm, das den Verbrauch, die Tankgröße, die anfängliche Füllung des Tanks, die Länge der zu fahrenden Strecke und eine Liste von Tankstellen mit jeweiligen Entfernungen und Preisen einliest und berechnet, an welchen Tankstellen getankt werden soll und jeweils wie viel Diesel.

Es muss unbedingt so selten wie möglich getankt werden, selbst wenn häufigeres Tanken billiger sein könnte. Die Anzahl der Tankvorgänge muss also minimal sein, und sie sollen so geplant werden, dass insgesamt möglichst wenig Geld ausgegeben wird. Der Tank darf am Ziel leer sein.

Wende dein Programm auf alle Beispiele an, die sich auf der BWINF-Website befinden.

#### Beispieldaten (`beispieldaten/fahrt1.txt` bis `fahrt5.txt`)

Verbrauch (l/100 km), Tankgröße (l), anfängliche Füllung (l), Streckenlänge (km), Anzahl der Tankstellen, dann je Tankstelle Entfernung vom Start (km) und Dieselpreis (Cent/l) – alles ganzzahlig.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zwei Ziele nacheinander:** Erst die kleinstmögliche Zahl der Tankstopps bestimmen, dann unter allen Plänen mit so vielen Stopps den billigsten suchen.
- **Greedy-Verfahren für die Stopps:** Wer so selten wie möglich tanken will, fährt immer bis zur *am weitesten entfernten* Tankstelle, die er noch erreicht, und tankt dort voll.
- **Tankregel (Austauschargument):** Ist die nächste Tankstelle, die man ansteuert, *nicht teurer* als die jetzige, tankt man nur so viel, wie man bis dorthin braucht. Ist sie *teurer*, tankt man hier voll. Mehr Möglichkeiten muss man nicht probieren.
- **Rekursion mit Gedächtnis (dynamische Programmierung):** Für jede Situation „an Tankstelle `i`, noch `k` Stopps erlaubt, so viel Diesel im Tank“ merken wir uns die günstigsten Restkosten. Ohne Gedächtnis würden dieselben Situationen unzählige Male neu berechnet.
- **Exakt rechnen:** Verbrauch und Tankgröße liefern Bruchzahlen (8 l auf 100 km). Mit Brüchen vermeiden wir Rundungsfehler, die den Vergleich zweier Kosten verfälschen würden.

**Nützliche Funktionen und Bibliotheken**
- `fractions.Fraction`: rechnet exakt mit Brüchen (`Fraction(55 * 100, 8)` ist genau 687,5). Mit `float(...)` bekommt man am Ende eine Dezimalzahl.
- `max(liste)`, Wörterbücher (`dict`) als Gedächtnis, Tupel als Schlüssel.

## Die Idee

**Schritt A: die wenigsten Stopps.** Mit der Anfangsfüllung `f` kommt man `f · 100 / Verbrauch` Kilometer weit. Reicht das bis zum Ziel, braucht man keinen Stopp. Sonst fahren wir zur entferntesten erreichbaren Tankstelle und tanken voll; ab dort kommt man `Tankgröße · 100 / Verbrauch` Kilometer weiter. Das wiederholen wir. Weiter fahren ist nie schlechter, deshalb ist die Zahl der Stopps so minimal.

**Schritt B: der billigste Plan mit genau so vielen Stopps.** Jetzt wählen wir Stopps und Tankmengen. Wir probieren von der aktuellen Tankstelle aus jede erreichbare nächste Tankstelle (und das Ziel) und rechnen für sie aus, was wir hier tanken müssen:

- Ist die nächste Tankstelle **billiger oder gleich teuer**, tanken wir nur so viel, dass wir sie gerade erreichen (oder gar nichts, wenn der Rest im Tank reicht).
- Ist sie **teurer**, tanken wir hier **voll**, denn hier ist der Diesel günstiger als dort.

Der Rest ist eine Rekursion: Kosten ab der nächsten Tankstelle mit einem Stopp weniger. Das Ziel behandeln wir wie eine Tankstelle mit Preis 0, an der man nicht tankt.

Die Zahl der verschiedenen Situationen ist kleiner, als man fürchtet: Der Rest im Tank bei Ankunft ist entweder 0 (genau reichend getankt), oder er ergibt sich aus einer vollen Füllung an einer früheren Tankstelle, oder er ist die Anfangsfüllung minus Strecke.

## Schritt 1: Einlesen

Die Datei enthält fünf Zahlen, danach je Tankstelle Entfernung und Preis. Wir lesen einfach *alle* Wörter der Datei in eine Liste von Zahlen; das ist egal, wie viele Leerzeichen oder Zeilenumbrüche dazwischen stehen. Wir sortieren die Tankstellen nach der Entfernung.

In [ ]:
def lies_fahrt(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zahlen = [int(w) for w in f.read().split()]
    verbrauch, tankgroesse, anfang, laenge, anzahl = zahlen[:5]
    stationen = []
    for i in range(anzahl):
        entfernung = zahlen[5 + 2 * i]
        preis = zahlen[6 + 2 * i]
        stationen.append((entfernung, preis))
    stationen.sort()
    return verbrauch, tankgroesse, anfang, laenge, stationen


NAMEN = ['beispieldaten/fahrt%d.txt' % i for i in range(1, 6)]
FAHRTEN = [lies_fahrt(name) for name in NAMEN]
for fahrt in FAHRTEN:
    verbrauch, tankgroesse, anfang, laenge, stationen = fahrt
    print(verbrauch, tankgroesse, anfang, laenge, len(stationen), 'Tankstellen, erste:', stationen[0])

8 55 22 1000 3 Tankstellen, erste: (100, 145)
24 274 273 10000 154 Tankstellen, erste: (24, 119)
30 160 160 800 14 Tankstellen, erste: (7, 142)
30 120 111 1000 17 Tankstellen, erste: (7, 115)
21 244 42 10000 171 Tankstellen, erste: (16, 133)


## Schritt 2: Die wenigsten Stopps

Die Reichweite in Kilometern ist `Diesel · 100 / Verbrauch`. Wir rechnen mit `Fraction`, damit nichts gerundet wird. `weite` ist die Stelle, bis zu der wir im Moment kommen. Solange das Ziel nicht erreicht ist, wählen wir die entfernteste Tankstelle `d <= weite`, die hinter unserer jetzigen Position liegt, tanken dort voll und aktualisieren `weite`. Gibt es keine, ist die Fahrt unmöglich.

In [ ]:
from fractions import Fraction


def wenigste_stopps(fahrt):
    verbrauch, tankgroesse, anfang, laenge, stationen = fahrt
    volle_reichweite = Fraction(tankgroesse * 100, verbrauch)
    weite = Fraction(anfang * 100, verbrauch)       # so weit kommen wir im Moment
    position = 0
    stopps = []
    while weite < laenge:
        kandidaten = [d for d, p in stationen if position < d <= weite]
        if len(kandidaten) == 0:
            return None                              # unmöglich
        position = max(kandidaten)
        stopps.append(position)
        weite = position + volle_reichweite
    return stopps


for i, fahrt in enumerate(FAHRTEN):
    stopps = wenigste_stopps(fahrt)
    print('fahrt%d: mindestens %d Stopps, z. B. bei km %s' % (i + 1, len(stopps), stopps[:6]))

fahrt1: mindestens 2 Stopps, z. B. bei km [100, 400]
fahrt2: mindestens 9 Stopps, z. B. bei km [1118, 2240, 3346, 4455, 5581, 6651]
fahrt3: mindestens 1 Stopps, z. B. bei km [465]
fahrt4: mindestens 2 Stopps, z. B. bei km [344, 706]
fahrt5: mindestens 9 Stopps, z. B. bei km [194, 1305, 2433, 3542, 4642, 5751]


Das sind genau die Stopp-Zahlen aus den Lösungshinweisen (2, 9, 1, 2, 9). Die gezeigten Stopps sind nur *ein* Plan mit so wenigen Halten; welche Tankstellen billig sind, wissen wir noch nicht.

## Schritt 3: Der billigste Plan mit höchstens `m` Stopps

`kosten(...)` liefert zu einer Situation die kleinsten Restkosten (in Cent) und den zugehörigen Plan. Die Parameter:

- `stat`: die Liste aller Orte `(Entfernung, Preis)`: der Start `(0, ...)`, alle Tankstellen, und das Ziel `(Länge, 0)`.
- `i`: Index des Ortes, an dem wir gerade sind. `k`: so viele Stopps sind noch erlaubt (einschließlich dem hier). `r`: Liter im Tank bei Ankunft.
- `gedaechtnis`: Wörterbuch mit den schon berechneten Situationen.

Am Start (`i == 0`) und wenn keine Stopps mehr übrig sind (`k == 0`) darf man nicht tanken. Dann kommen nur Orte in Frage, die man mit dem Rest im Tank (`need <= r`) erreicht. Sonst probieren wir alle Orte `j` hinter `i`, die höchstens eine Tankfüllung entfernt sind, und wenden die Tankregel an. Der Preis des Ziels ist 0, es ist also immer „billiger“, und wir tanken nur das Nötige.

In [ ]:
UNMOEGLICH = 10 ** 12


def kosten(stat, verbrauch, tankgroesse, i, k, r, gedaechtnis):
    """Kleinste Restkosten (Cent) und Plan (Tupel aus (km, Liter, Preis)) ab Ort i."""
    schluessel = (i, k, r)
    if schluessel in gedaechtnis:
        return gedaechtnis[schluessel]
    d, p = stat[i]
    ziel = len(stat) - 1
    bester = (UNMOEGLICH, ())
    darf_tanken = (i != 0 and k >= 1)
    for j in range(i + 1, len(stat)):
        need = Fraction((stat[j][0] - d) * verbrauch, 100)     # Liter bis zum Ort j
        if need > tankgroesse:
            break                                              # zu weit, auch alle weiteren
        if darf_tanken:
            if stat[j][1] <= p:
                kauf = max(Fraction(0), need - r)              # nur so viel wie nötig
            else:
                kauf = tankgroesse - r                         # voll tanken
            k2 = k - 1
        else:
            if need > r:
                break                                          # mit dem Rest nicht erreichbar
            kauf = Fraction(0)
            k2 = k
        if j == ziel:
            rest, plan = 0, ()
        else:
            rest, plan = kosten(stat, verbrauch, tankgroesse, j, k2, r + kauf - need, gedaechtnis)
            if rest >= UNMOEGLICH:
                continue
        summe = rest + kauf * p
        if summe < bester[0]:
            if kauf > 0:
                plan = ((d, kauf, p),) + plan
            bester = (summe, plan)
    gedaechtnis[schluessel] = bester
    return bester


def bester_plan(fahrt, m):
    verbrauch, tankgroesse, anfang, laenge, stationen = fahrt
    stat = [(0, 10 ** 6)] + stationen + [(laenge, 0)]
    gedaechtnis = {}
    summe, plan = kosten(stat, verbrauch, tankgroesse, 0, m, Fraction(anfang), gedaechtnis)
    return summe, plan, len(gedaechtnis)

## Schritt 4: Ausgabe und alle Beispiele

Wir geben für jeden Stopp Kilometer, Preis und Menge aus, dazu die Gesamtkosten in Euro. Die Anfangsfüllung kostet nichts (sie ist schon im Tank), Kosten und getankte Liter zählen also erst ab dem ersten Stopp.

In [ ]:
def loese(fahrt, ausfuehrlich=True):
    m = len(wenigste_stopps(fahrt))
    summe, plan, zustaende = bester_plan(fahrt, m)
    if ausfuehrlich:
        for d, liter, p in plan:
            print('  bei km %5d: %7.2f l zu %d ct/l' % (d, float(liter), p))
    liter_gesamt = sum(liter for d, liter, p in plan)
    return m, len(plan), float(summe) / 100, float(liter_gesamt), zustaende, plan


m, anzahl, euro, liter, zustaende, plan = loese(FAHRTEN[0])
print('Stopps:', anzahl, ' Kosten: %.2f Euro' % euro, ' getankt: %.1f l' % liter)

  bei km   100:   10.00 l zu 145 ct/l
  bei km   400:   48.00 l zu 140 ct/l
Stopps: 2  Kosten: 81.70 Euro  getankt: 58.0 l


Beispiel 1: Auf 1000 km braucht das Wohnmobil (8 l auf 100 km) 80 l, im Tank sind 22 l. Es muss also 58 l tanken. Mit zwei Stopps gelingt das für 81,70 Euro.

Jetzt alle fünf Beispiele. Für die langen Beispiele geben wir nur die Zusammenfassung aus. `Zustände` ist die Zahl der Situationen im Gedächtnis.

In [ ]:
ERGEBNISSE = []
print('Nr  Stopps  Kosten (Euro)  getankt (l)  Zustände')
for i, fahrt in enumerate(FAHRTEN):
    m, anzahl, euro, liter, zustaende, plan = loese(fahrt, ausfuehrlich=False)
    ERGEBNISSE.append((m, anzahl, euro, liter, plan))
    print('%2d  %6d  %13.2f  %11.0f  %8d' % (i + 1, anzahl, euro, liter, zustaende))

Nr  Stopps  Kosten (Euro)  getankt (l)  Zustände
 1       2          81.70           58         4
 2       9        2530.33         2127     10771
 3       1          99.20           80        22
 4       2         236.88          189       124
 5       9        2494.32         2058     11618


Das stimmt mit der Tabelle der Lösungshinweise überein (optimale Kosten 81,70 / 2530,33 / 99,20 / 236,88 / 2494,32 Euro, Stopps 2 / 9 / 1 / 2 / 9, Treibstoff 58 / 2127 / 80 / 189 / 2058 Liter).

Als Beispiel die ausführliche Ausgabe für Beispiel 3 (ein Stopp) und Beispiel 4 (zwei Stopps):

In [ ]:
for nr in [3, 4]:
    print('Beispiel', nr)
    m, anzahl, euro, liter, zustaende, plan = loese(FAHRTEN[nr - 1])
    print('  Kosten: %.2f Euro, %d Stopps' % (euro, anzahl))

Beispiel 3
  bei km   465:   80.00 l zu 124 ct/l
  Kosten: 99.20 Euro, 1 Stopps
Beispiel 4
  bei km   264:   88.20 l zu 120 ct/l
  bei km   607:  100.80 l zu 130 ct/l
  Kosten: 236.88 Euro, 2 Stopps


Für Beispiel 2 und 5 (9 Stopps, über 150 Tankstellen) ist die Ausgabe der einzelnen Stopps lang; wir zeigen nur die ersten drei.

In [ ]:
for nr in [2, 5]:
    plan = ERGEBNISSE[nr - 1][4]
    print('Beispiel', nr, ': %d Stopps, die ersten drei:' % len(plan))
    for d, liter, p in plan[:3]:
        print('  bei km %5d: %7.2f l zu %d ct/l' % (d, float(liter), p))

Beispiel 2 : 9 Stopps, die ersten drei:
  bei km  1118:  269.32 l zu 115 ct/l
  bei km  1922:  192.96 l zu 117 ct/l
  bei km  2762:  184.64 l zu 119 ct/l
Beispiel 5 : 9 Stopps, die ersten drei:
  bei km   107:  209.58 l zu 116 ct/l
  bei km  1198:  244.00 l zu 115 ct/l
  bei km  2344:  229.76 l zu 117 ct/l


## Für Neugierige

- **Warum reicht die Tankregel?** Wer an einer Tankstelle steht und weiß, welche Tankstelle er als nächste ansteuert, kann nichts Besseres tun als: Ist die nächste nicht teurer, so wenig wie möglich hier tanken (dort ist der Diesel höchstens genauso teuer). Ist sie teurer, so viel wie möglich hier tanken. Dass wir *alle* erreichbaren nächsten Tankstellen ausprobieren, deckt alle Pläne ab.
- **Laufzeit.** Das Gedächtnis hat in den großen Beispielen nur gut 10 000 Einträge (siehe Spalte „Zustände“). Die Zahl der möglichen Resttankfüllungen ist nämlich durch die Zahl der Tankstellen beschränkt. Die Lösungshinweise erwähnen noch eine Beschleunigung durch Weglassen von Tankstellen, von denen aus das Ziel mit den restlichen Stopps nicht mehr erreichbar ist (hier nicht nötig).
- **Gegenprobe Esra.** Wer an den Stopps des Greedy-Verfahrens aus Schritt 2 immer volltankt (Esras Vorschlag), gibt mehr aus. Die folgende Zelle rechnet das nach; der Tank ist dann am Ziel nicht leer, sondern der Diesel wurde teuer bezahlt.

In [ ]:
def volltanken_kosten(fahrt):
    """Kosten, wenn an jedem Stopp von wenigste_stopps voll getankt wird."""
    verbrauch, tankgroesse, anfang, laenge, stationen = fahrt
    preis = {d: p for d, p in stationen}
    orte = [0] + wenigste_stopps(fahrt) + [laenge]
    tank = Fraction(anfang)
    gesamt = 0
    for a, b in zip(orte, orte[1:]):
        if a != 0:
            gesamt = gesamt + (tankgroesse - tank) * preis[a]    # volltanken
            tank = Fraction(tankgroesse)
        tank = tank - Fraction((b - a) * verbrauch, 100)
    return float(gesamt) / 100


for i, fahrt in enumerate(FAHRTEN):
    print('Beispiel %d: immer volltanken %.2f Euro, optimal %.2f Euro' % (i + 1, volltanken_kosten(fahrt), ERGEBNISSE[i][2]))

Beispiel 1: immer volltanken 93.05 Euro, optimal 81.70 Euro
Beispiel 2: immer volltanken 2996.23 Euro, optimal 2530.33 Euro
Beispiel 3: immer volltanken 172.98 Euro, optimal 99.20 Euro
Beispiel 4: immer volltanken 286.79 Euro, optimal 236.88 Euro
Beispiel 5: immer volltanken 2814.16 Euro, optimal 2494.32 Euro


## Zum Schluss: Urlaubsfahrt mit MiniZinc

Auch diese Aufgabe lässt sich in MiniZinc beschreiben, ohne den Algorithmus zu erfinden: Wir sagen, was ein gültiger Tankplan ist, und minimieren die Kosten. Die Zahl der Stopps `m` bestimmen wir wie oben mit dem Greedy-Verfahren und geben sie dem Modell vor (der Solver soll nur noch den billigsten Plan finden).

**Modell.**

- Damit alles ganzzahlig bleibt, rechnen wir den Diesel in Einheiten zu **1/100 Liter**. Dann verbraucht jeder Kilometer genau `v` Einheiten, und eine Tankfüllung hat `G = Tankgröße · 100` Einheiten.
- `kauf[i]` ist die Menge, die am Ort `i` getankt wird, `ab[i]` die Füllung beim Wegfahren. Die Orte sind der Start (Index 0), die Tankstellen (1 bis `z`) und das Ziel (`z+1`).
- Die Füllung beim Wegfahren ist die Füllung beim Wegfahren am vorigen Ort, minus Verbrauch für die Strecke, plus Gekauftes. Beim Ankommen darf der Tank nicht negativ sein (`>= 0`).
- `kauf[i] > 0 -> stopp[i]`: Wer tankt, hält an. Die Zahl der Stopps ist höchstens `m`. Die Kosten sind die Summe aus `kauf[i] * p[i]`, in 1/100 Liter mal Cent pro Liter, also in 1/10 000 Euro.

```minizinc
int: z;                                       % Anzahl der Tankstellen
int: v;                                       % Verbrauch: Einheiten pro km (1 Einheit = 1/100 Liter)
int: G;                                       % Tankgröße in Einheiten
int: F0;                                      % Anfangsfüllung in Einheiten
int: m;                                       % höchstens so viele Stopps
array[0..z+1] of int: d;                      % Entfernung: 0 ist der Start, z+1 das Ziel
array[0..z+1] of int: p;                      % Preis in Cent pro Liter

array[0..z+1] of var 0..G: kauf;              % an diesem Ort getankte Einheiten
array[0..z+1] of var 0..G: ab;                % Einheiten im Tank beim Wegfahren
array[0..z+1] of var bool: stopp;             % wird hier getankt?

constraint ab[0] = F0 /\ kauf[0] = 0 /\ kauf[z+1] = 0;
% beim Ankommen darf der Tank nicht leer sein, beim Wegfahren ist er voller um die gekaufte Menge
constraint forall(i in 1..z+1)(ab[i-1] - v * (d[i] - d[i-1]) >= 0
                               /\ ab[i] = ab[i-1] - v * (d[i] - d[i-1]) + kauf[i]);
constraint forall(i in 0..z+1)(kauf[i] > 0 -> stopp[i]);
constraint sum(stopp) <= m;

var int: kosten = sum(i in 0..z+1)(kauf[i] * p[i]);
solve minimize kosten;

output ["Kosten: ", show(kosten), "\n",
        "Tankstellen bei km ", show([d[i] | i in 0..z+1 where fix(kauf[i]) > 0]), "\n",
        "Einheiten: ", show([fix(kauf[i]) | i in 0..z+1 where fix(kauf[i]) > 0])];
```

**Daten** für das erste Beispiel (die Python-Zelle `als_dzn` unten baut sie aus den eingelesenen Werten):

```minizinc
z = 3;
v = 8;
G = 5500;
F0 = 2200;
m = 2;
d = array1d(0..4, [0, 100, 400, 900, 1000]);
p = array1d(0..4, [0, 145, 140, 122, 0]);
```

**Ausgabe:**

```
Kosten: 817000
Tankstellen bei km [100, 400]
Einheiten: [1000, 4800]
----------
==========
```

Die Kosten 817000 sind 81,70 Euro. Ergebnisse für die Beispiele, die der Solver schafft (Solver **Gecode**, jeweils Sekundenbruchteile):

```
Beispiel 1  ->  817000  (81.70 Euro)
Beispiel 3  ->  992000  (99.20 Euro)
Beispiel 4  ->  2368800  (236.88 Euro)
```

Das sind dieselben Kosten wie bei unserer Rekursion.

**Grenze.** Bei den Beispielen 2 und 5 (jeweils über 150 Tankstellen, 9 Stopps) liefert MiniZinc in unseren Tests innerhalb von 30 Sekunden kein bewiesenes Optimum, weder mit Gecode noch mit Chuffed oder CP-SAT. Das Modell weiß nichts von der Tankregel und muss tatsächlich alle Mengen und Stopps durchprobieren; unser Verfahren nutzt die Tankregel und braucht nur gut 10 000 Situationen. Für dieses Problem ist der selbst geschriebene Algorithmus also deutlich besser als das einfache Modell.

In [ ]:
def als_dzn(fahrt, m):
    verbrauch, tankgroesse, anfang, laenge, stationen = fahrt
    orte = [(0, 0)] + stationen + [(laenge, 0)]
    zeilen = ['z = %d;' % len(stationen), 'v = %d;' % verbrauch, 'G = %d;' % (tankgroesse * 100),
              'F0 = %d;' % (anfang * 100), 'm = %d;' % m,
              'd = array1d(0..%d, %s);' % (len(stationen) + 1, [d for d, p in orte]),
              'p = array1d(0..%d, %s);' % (len(stationen) + 1, [p for d, p in orte])]
    return '\n'.join(zeilen)


print(als_dzn(FAHRTEN[0], 2))

z = 3;
v = 8;
G = 5500;
F0 = 2200;
m = 2;
d = array1d(0..4, [0, 100, 400, 900, 1000]);
p = array1d(0..4, [0, 145, 140, 122, 0]);
